# Notebook 3: Classification with the scikit-learn Breast Cancer Dataset
*Author: Sara Speelman; 2026-08-11*

In this notebook, you will apply a full machine learning workflow to a **classification problem** using the breast cancer dataset from scikit-learn. You will use different models in the family of the decision tree:
- baseline model
- decision tree
- random forest
- XGBoost models

Furthermore, we will add **feature selection** to our workflow, and practise hyperparameter tuning some more.

Learning outcomes:
- create a correct train/val/test set for a classification problem
- training and evaluating a model for a **classification problem in a medical application**
- using scikit-learn's Pipeline for the model workflow, with **feature selection** and hyperparameter tuning

For this classification problem, we use the __[Breast Cancer Wisconsin (Diagnostic) dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-dataset)__ from scikit-learn. It contains 569 samples, derived from digitized images of fine needle aspirates (FNA). Each sample includes 30 features describing cell nuclei characteristics, and a binary target indicating whether the tumor is malignant (0) or benign (1). 

In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, roc_auc_score, roc_curve, f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import random

random_state = 42
random.seed(random_state)

In [ ]:
# DEMO
# Load the breast cancer dataset, and create a DataFrame for the features and a Series for the target variable
data = load_breast_cancer(as_frame=True)

X = data.data
y = data.target

## 1. Global Data Inspection & Cleaning

- Inspect data types, missing values, and duplicates.
- Clean data as needed.
- Inspect class distribution/imbalance.

In [ ]:
# EXERCISE
# Print the description of the dataset, using .DESCR


In [ ]:
# EXERCISE
# Print the shape of X and y


In [ ]:
# EXERCISE
# Show the first few rows of the features dataset


In [ ]:
# EXERCISE
# Show the first few rows of the target variable


In [ ]:
# EXERCISE
# Check Dtype of the features
print("\nDtypes summary:")


In [ ]:
# EXERCISE
# Check whether there are any NaNs over the whole dataset (same as isnull)


In [ ]:
# EXERCISE
# Check for duplicate columns



# Check for duplicate rows


In [ ]:
# EXERCISE
# Check whether there is class imbalance
print('Target Distribution (proportion)')


Let's not forget the basics: do we understand our problem? What do the class labels mean? Is malignant a zero or a one?  
The description mentions there are more benign than malignant cases, so benign must be class 1. We can also get this information directly from the data object

In [ ]:
# EXERCISE
# Print the target class names


## 2. Train/Validation/Test Split

We will split our dataset in a **training set (crossvalidation for hyperparam tuning)**, a **validation set (model selection)** and a **test set (performance estimate on unseen data)**.  
Since we have mild class imbalance, how can we ensure our different sets are representative for our data distribution? Have a look at the __[documentation of train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)__ to check which parameter can help you with this.

In [ ]:
# EXERCISE
# Split data into training and testing sets using stratified train_test_split


How do our data splits look now? A plot would help to get more intuition:

In [ ]:
# DEMO
# Plot bar plots of the target variable distribution in the train, validation, and test sets
# HINT:You could use seaborn's countplot 
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
sns.countplot(x=y_train, ax=axes[0])
axes[0].set_title('Train Set')
sns.countplot(x=y_val, ax=axes[1])
axes[1].set_title('Validation Set')
sns.countplot(x=y_test, ax=axes[2])
axes[2].set_title('Test Set')
plt.tight_layout()
plt.show()

## 3. Exploratory Data Analysis (EDA)

### 3.1 Summary Statistics

In [ ]:
# EXERCISE
# Summary statistics of the features in the training dataset


What can you conclude about the feature scales?

### 3.2 Data Visualization

It's important to vizualize our data to get a better understanding of the dataset we are working with. For a classification problem, a number of things can provide more insight, e.g.
- scatter plots of feature pairs
- Violin plots, histograms and box plots to inspect distributions and outliers

In [ ]:
# DEMO
# Pick feature pairs and plot them as scatter plots with points colored by class
feature_pairs = [
    ("mean radius", "mean texture"),
    ("mean area", "mean concavity"),
    ("worst radius", "worst concavity")
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (feature1, feature2) in enumerate(feature_pairs):
    sc = axes[i].scatter(
        X_train[feature1],
        X_train[feature2],
        c=y_train,
        cmap="viridis",
        alpha=0.8,
        edgecolors="k",
        linewidths=0.2
    )
    axes[i].set_xlabel(feature1)
    axes[i].set_ylabel(feature2)
    axes[i].set_title(f"{feature1} vs {feature2}")

# Shared legend (0 = malignant, 1 = benign)
handles, _ = sc.legend_elements()
fig.legend(handles, data.target_names, title="Class", loc="upper right")

plt.tight_layout()
plt.show()

Do you see a rough line that could seperate the classes?  
Is there an overlap region? Those points will be hard for the classifier.

Next, we'll check whether we have outliers. We can see the ranges of the features in the summary statistics. Let's make plots of some features with large ranges.  

A **violin plot** shows the distribution of a variable by combining a boxplot with a smoothed density shape. The wider parts of the violin indicate where more data points are concentrated, while the thinner parts show where values are less common.

In [ ]:
# DEMO
# Compute feature ranges from X_train.describe() and show the top 5 features with the largest ranges
desc = X_train.describe()
feature_ranges = desc.loc["max"] - desc.loc["min"]

top_5_ranges = feature_ranges.sort_values(ascending=False).head(5)
print(top_5_ranges)

In [ ]:
# DEMO
# Violin plots, Histograms + KDE, and Box plots for the 5 features with the largest ranges

top_features = top_5_ranges.index.tolist()

fig, axes = plt.subplots(3, 5, figsize=(24, 12))

# Row 1: Violin plots split by class
for i, feature in enumerate(top_features):
    sns.violinplot(
        data=X_train.assign(target=y_train),
        x="target",
        y=feature,
        ax=axes[0, i],
        inner="quartile"
    )
    axes[0, i].set_title(feature)
    axes[0, i].set_xlabel("Class")
    axes[0, i].set_xticks([0, 1])
    axes[0, i].set_xticklabels(data.target_names)

# Row 2: Histograms + KDE
for i, feature in enumerate(top_features):
    sns.histplot(
        X_train[feature],
        kde=True,
        ax=axes[1, i]
    )
    axes[1, i].set_title(feature)
    axes[1, i].set_xlabel(feature)

# Row 3: Box plots split by class
for i, feature in enumerate(top_features):
    sns.boxplot(
        data=X_train.assign(target=y_train),
        x="target",
        y=feature,
        ax=axes[2, i],
        hue="target",
        legend=False
    )
    axes[2, i].set_title(feature)
    axes[2, i].set_xlabel("Class")
    axes[2, i].set_xticks([0, 1])
    axes[2, i].set_xticklabels(data.target_names)

plt.tight_layout()
plt.show()


What do these box plots tell us about feature distributions and outliers? Do we have to scale our data here?  
Have a look at some possible scalers:

- __[StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)__
- __[RobustScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html)__
- __[MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)__  

A common approach is to **standardize** each feature with `StandardScaler`:

- centering it: subtract the mean
- scaling it: divide by the standard deviation

This gives features with approximately mean 0 and variance 1.
If your data contains many outliers, scaling using the mean and variance of the data is likely to not work very well. In these cases, you can use RobustScaler as a drop-in replacement instead. It uses more robust estimates for the center and range of your data. __[source](https://scikit-learn.org/stable/modules/preprocessing.html#preprocessing-scaler)__

We invite you to have a look at a __[comparison of the effect of different scalers on data with outliers](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_all_scaling.html#plot-all-scaling-standard-scaler-section)__

### 3.3 Correlation matrix

Make a correlation matrix of our training set features to capture linear relationships. Why is this relevant?  
Make sure you are familiar with __[multicollinearity](https://www.geeksforgeeks.org/machine-learning/multicollinearity-in-data/)__ and related issues.

In [ ]:
# EXERCISE
# Plot a correlation matrix of the training features
corr = ...

What do we learn from our correlation matrix? Would we feed all our features to the model? 

How could you use the correlation information to filter features? Is this relevant in tree-based models?

## 4. Feature Selection
Next to correlation with other features, we explore the relevance of the candidate features in predicting the target.

In this section, we explore two different feature selection methods: **SelectKBest** and **Feature Importance**. We invite you to explore other methods yourself, for example, don't hesitate to delve into __[Recursive feature elimination (RFE)](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html)__, or dimensionality reduction techniques such as **PCA**.

As an example, we will select the best 10 features with these methods. This section is mainly to help you get a feeling of these methods, but won't actually be used by our model. Instead, we will add feature selection (with SelectKBest) in our Pipeline below. Integrating **feature selection in the Pipeline** helps to avoid data leakage.

### 4.1 SelectKBest demo
**SelectKBest** selects the top k features that have the strongest relationship with the target variable, based on a **statistical test**. We will use **mutual information** here, applying the theory you have seen in class.

Read the docs: https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html 

In [ ]:
# DEMO
# Select the top 10 features using mutual information
from sklearn.feature_selection import SelectKBest, mutual_info_classif
selector = SelectKBest(score_func=mutual_info_classif, k=10)
selector.fit(X_train, y_train)
selected_features = X_train.columns[selector.get_support()]
print('Selected features (SelectKBest): \n', selected_features.tolist())

### 4.2 Feature Importances demo
Tree-based models like Random Forests can provide an estimate of feature importance, which reflects how useful each feature was in the construction of the model. We can use the **feature_importances_** attribute of the **RandomForestClassifier** object of scikit-learn. In Random Forests, impurity-based feature importance measures how much a feature helps reduce node impurity during tree construction. 
Below, we fit a Random Forest and visualize the top features. 

Have a look at **permutation importance** on your own. This one measures how much model performance decreases when the values of that feature are randomly shuffled after training.

In [ ]:
# DEMO
# Feature importance using Random Forest
from sklearn.ensemble import RandomForestClassifier

# Create and fit the model
rf = RandomForestClassifier(random_state=random_state)
rf.fit(X_train, y_train)

# Retrieve feature importances thourgh the feature_importances_ attribute
importances = rf.feature_importances_

# Print the top 10 features
indices = np.argsort(importances)[::-1]
print('Top 10 features by importance:')
for f in range(10):
    print(f"{f + 1}. {X_train.columns[indices[f]]} ({importances[indices[f]]:.4f})")

# Plot feature importances
plt.figure(figsize=(8,4))
plt.title('Feature Importances (Random Forest)')
sns.barplot(x=importances[indices[:10]], y=X_train.columns[indices[:10]])
plt.show()

## 5. Model Training
In this section, we will train a Random Forest (7.1) and an XGBoost classifier (7.2), with SelectKBest for feature selection. To help you get started, read the example of the decision tree pipeline.

In [ ]:
# DEMO
# Create a single cross-validation splitter, shared by all models below, for a fair comparison and reproducibility
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

In [ ]:
# DEMO
# Decision Tree pipeline with GridSearchCV
# Create a pipeline with scaling, feature selection, and classifier
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=mutual_info_classif, k=10)),
    ('clf', DecisionTreeClassifier(random_state=random_state))
])
# Define parameter grid for GridSearchCV
param_grid = {
    'clf__max_depth': [3, 5, 7, None],
    'clf__min_samples_split': [2, 5, 10]
}
# Perform Grid Search with cross-validation
grid = GridSearchCV(pipe, param_grid, cv=cv_splitter, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

# Print best parameters and best cross-validation score
print('Best parameters:', grid.best_params_)
print('Best cross-val accuracy:', round(grid.best_score_, 4))

Set `n_jobs=-1` to use all available CPU cores to run the search in parallel. Otherwise `GridSearchCV` can take a long time to run, for larger parameter grids.

### 5.0 Baseline model
As a dummy baseline to compare or model performance to, we use a majority classifier.

In [ ]:
# DEMO
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

# Baseline: majority classifier (predicts most frequent class)

majority_clf = DummyClassifier(strategy="most_frequent")
majority_clf.fit(X_train, y_train)

### 5.1 Random Forest Classifier with RandomizedSearchCV

We will use __[RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)__ here, which samples a fixed number (n_iter, 10 by default) of parameter settings from the specified distributions, making it more efficient for larger search spaces. In other words: `GridSearchCV` is more exhaustive, while `RandomizedSearchCV` is usually more efficient.  
Beware, with RandomizedSearchCV we are **sampling from a parameter distribution** instead of looping through all parameters in a parameter grid. We must thus define parameter distributions here.

A `RandomForestClassifier` combines many decision trees and makes a final prediction based on all of them rather than relying on a single tree. This usually makes it more stable and less prone to overfitting than one decision tree alone. Read the __[documentation of the RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)__ to see which important hyperparameters we have to tune. Think about what parameter distributions would make sense.

In [ ]:
# EXERCISE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# Create a Random Forest pipeline with RandomizedSearchCV
rf_pipe = ...
# Define parameter distribution for RandomizedSearchCV
param_dist_rf = ...
# Perform Randomized Search with cross-validation
rf_random_search = ...
# Fit the RandomizedSearchCV object to the training data


# Print best parameters and best cross-validation score


**Note**: I randomly chose k=10 as amount of features. But we could optimise this as well. Try yourself whether you can adapt the code cell above to make k a parameter in our in our parameter distribution as well.

### 5.2 XGBoost Classifier with RandomizedSearchCV
**XGBClassifier** is an ML model that uses 'eXtreme Gradient Boosting'. It sequentially builds multiple decision trees, with each tree correcting the errors of the previous ones. The final result is a combination of the result of the individual trees. 

Read the __[documentation of XGBoost](https://xgboost.readthedocs.io/en/stable/parameter.html)__ to understand which hyperparameters we should tune. Or the __[GeeksforGeeks page](https://www.geeksforgeeks.org/machine-learning/xgbclassifier/)__, as it's an easier introduction.

In [ ]:
# EXERCISE
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from scipy.stats import randint, uniform, loguniform

# Create an XGBoost pipeline
xgb_pipe = ...
param_dist_xgb = ...

In [ ]:
# EXERCISE
# use RandomizedSearchCV to find the best hyperparameters for XGBClassifier (with our defined param_dist)
xgb_pipe_random_search = ...
# Fit the RandomizedSearchCV to the training data


# Print best parameters and best cross-validation score


## 6. Model selection
We will compare the performance of our 2 tree based models on the validation set, and choose the one that performs best.

In [ ]:
# DEMO
# Compare validation accuracy of RF (random search) and XGB (random search)
# Select the best_estimator (the refit model on the whole training set) from each model + hyperparameter tuning method
best_rf = rf_random_search.best_estimator_
best_xgb_rand = xgb_pipe_random_search.best_estimator_

# Predict on validation set
y_val_dummy = majority_clf.predict(X_val)
y_val_pred_rf = best_rf.predict(X_val)
y_val_pred_xgb_rand = best_xgb_rand.predict(X_val)

# Compare validation accuracy of RF (random search), XGB (random search), and XGB (grid search)
# Use accuracy_score from sklearn.metrics to compute validation accuracies
acc_dummy = accuracy_score(y_val, y_val_dummy)
acc_rf = accuracy_score(y_val, y_val_pred_rf)
acc_xgb_rand = accuracy_score(y_val, y_val_pred_xgb_rand)

# Print the accuracy scores
print('Validation Accuracy (Dummy):', round(acc_dummy, 4))
print('Validation Accuracy (RF):', round(acc_rf, 4))
print('Validation Accuracy (XGB):', round(acc_xgb_rand, 4))

# Select the best model based on validation accuracy
if acc_rf > acc_xgb_rand:
    best_model = best_rf
    print("Best model: Random Forest")
else:
    best_model = best_xgb_rand
    print("Best model: XGBoost")

In [ ]:
# EXERCISE
# Select the model with the highest validation accuracy


## 7. Model Evaluation
Evaluate the performance of the model on unseen data. What is a suitable **performance metric** for this problem? Should we have used another performance metric in our cross-validation?   
Familiarize yourself with the following performance metrics:

- __[accuracy](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)__: fraction of correct predictions
- __[precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)__: **tp / (tp + fp)**: ability of a classifier not to label a negative sample as positive
- __[recall](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)__: **tp / (tp + fn)**: ability of the classifier to find all the positive samples
- __[F1 score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)__: **(2x tp)/ (2 x tp + fp + fn)**: a harmonic mean of the precision and recall, where an F1 score reaches its best value at 1 and worst score at 0. The relative contribution of precision and recall to the F1 score are equal.

In [ ]:
# EXERCISE
# Evaluate best model on test set
y_pred = ...

In [ ]:
# EXERCISE
# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(...)
...

In [ ]:
# EXERCISE
# Print evaluation metrics
# Include accuracy, confusion matrix, precision, recall
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.2f}")
print(f"Precision: {precision_score(y_test, y_pred, pos_label=0):.2f}")  # pos_label=0 because we want to compute precision for the positive class (malignant)
print(f"Recall: {...}")
print(f"F1-score: {...}")

# Print classification report
print('Classification Report:')
print(...)

What can you conclude about the scores? What should you do when you have such high performances? 

In [ ]:
# DEMO
# Calculate AUC-ROC and plot ROC curve
y_proba = best_model.predict_proba(X_test)[:,0]
fpr, tpr, thresholds = roc_curve(y_test, y_proba, pos_label=0)
auc_roc = 1- roc_auc_score(y_test, y_proba)
print('AUC-ROC:', round(auc_roc, 4))

# Plot AUC-ROC
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {auc_roc:.4f})')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

What does the ROC curve tell us? When is this metric useful? Is is suitable for our current medical application? Do we care evenly about false positives and false negatives?

**Disclaimer**: GenAI was used when creating this Notebook.